# 04c -- dim_dynasty_metric (metric_key index)

**Purpose:** Curated dimension for the `metric_key` field of
`fact_dynasty_ranking_metrics`. Lets Power BI use metrics as a **matrix column
axis** (`metric_label`) with **`metric_order`** controlling left-to-right flow
(set `metric_label`'s *Sort by column* = `metric_order`). `metric_group` gives an
optional column-group level; `direction` drives conditional formatting.

This is a hand-maintained transformer/seed (like `dim_position`): edit the SEED to
relabel, regroup, or reorder. The validation cell flags any metric_key present in
the fact but missing here (add a row) or defined here but unused.

**Output:** `data/dim_dynasty_metric.parquet` (PK `metric_key`).
Run with CWD = repo root.

In [5]:
# ---- Setup & Config ---------------------------------------------------------
from pathlib import Path
import pandas as pd

import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG

TABLE = "dim_dynasty_metric"
print("building", TABLE)

building dim_dynasty_metric


In [ ]:
# ---- Curated seed -----------------------------------------------------------
# (metric_key, metric_label, metric_group, metric_order, value_type, direction)
# direction: "up" higher=better | "down" lower=better | "neutral".
# metric_order is grouped in 10s with gaps so you can insert without renumbering.
SEED = [
    # Value
    ("value",                   "KTC Value",            "Value",      10, "num",  "up"),
    ("ds_value",                "DS 3D Value+",         "Value",      11, "num",  "up"),
    # Market
    ("adp",                     "ADP",                  "Market",     20, "num",  "down"),
    ("startup_adp",             "Startup ADP",          "Market",     21, "num",  "down"),
    ("avg_auction_pct",         "Auction %",            "Market",     22, "num",  "up"),
    ("startup_avg_auction_pct", "Startup Auction %",    "Market",     23, "num",  "up"),
    ("std_liquidity",           "Liquidity",            "Market",     24, "num",  "neutral"),
    # Tier (1 = best)
    ("overall_tier",            "Overall Tier",         "Tier",       30, "num",  "down"),
    ("positional_tier",         "Positional Tier",      "Tier",       31, "num",  "down"),
    # Projections (DynastySharks, fantasy points)
    ("proj_1yr",                "Proj 1-Yr",            "Projection", 40, "num",  "up"),
    ("proj_3yr",                "Proj 3-Yr",            "Projection", 41, "num",  "up"),
    ("proj_5yr",                "Proj 5-Yr",            "Projection", 42, "num",  "up"),
    ("proj_10yr",               "Proj 10-Yr",           "Projection", 43, "num",  "up"),
    # Consensus distribution (FantasyPros, rank units -> lower better)
    ("avg",                     "FP Avg Rank",          "Consensus",  50, "num",  "down"),
    ("best",                    "FP Best Rank",         "Consensus",  51, "num",  "down"),
    ("worst",                   "FP Worst Rank",        "Consensus",  52, "num",  "down"),
    ("stddev",                  "FP Std Dev",           "Consensus",  53, "num",  "down"),
    # Trends (KTC)
    ("overall_trend",           "Overall Trend",        "Trend",      60, "num",  "up"),
    ("positional_trend",        "Pos. Trend",           "Trend",      61, "num",  "up"),
    ("overall_7day_trend",      "Overall Trend (7d)",   "Trend",      62, "num",  "up"),
    ("positional_7day_trend",   "Pos. Trend (7d)",      "Trend",      63, "num",  "up"),
    # Crowd (KTC rating game)
    ("kept",                    "Kept",                 "Crowd",      70, "num",  "up"),
    ("traded",                  "Traded",               "Crowd",      71, "num",  "neutral"),
    ("cut",                     "Cut",                  "Crowd",      72, "num",  "down"),
    # Notes
    ("analysis",                "Analysis",             "Notes",      90, "text", "neutral"),
]

dim = pd.DataFrame(SEED, columns=[
    "metric_key", "metric_label", "metric_group", "metric_order", "value_type", "direction"])
assert dim["metric_key"].is_unique, "metric_key must be unique (PK)"
assert dim["metric_order"].is_unique, "metric_order should be unique for a clean sort"
print(f"{len(dim)} metric definitions")

25 metric definitions


In [7]:
# ---- Validate against the fact, then write ----------------------------------
fact_path = f"{CFG.data_dir}/fact_dynasty_ranking_metrics.parquet"
if Path(fact_path).exists():
    fact_keys = set(pd.read_parquet(fact_path, columns=["metric_key"])["metric_key"].unique())
    seed_keys = set(dim["metric_key"])
    missing = sorted(fact_keys - seed_keys)   # in fact, not defined -> ADD a row
    unused  = sorted(seed_keys - fact_keys)   # defined, not (yet) in fact -> ok
    if missing:
        print(f"[WARN] {len(missing)} metric_key in fact but missing from dim (add rows): {missing}")
    else:
        print(f"[ok] all {len(fact_keys)} fact metric_keys are defined in the dim")
    if unused:
        print(f"[info] {len(unused)} defined-but-unused (future sources): {unused}")
else:
    print("[info] fact_dynasty_ranking_metrics not found — skipping coverage check")

out = f"{CFG.data_dir}/{TABLE}.parquet"
dim.to_parquet(out, index=False)
print(f"[ok] wrote {len(dim)} rows -> {out}")

[info] fact_dynasty_ranking_metrics not found — skipping coverage check
[ok] wrote 25 rows -> data/dim_dynasty_metric.parquet


In [8]:
# ---- Preview (matrix column flow = metric_order) ----------------------------
print(dim.sort_values("metric_order").to_string(index=False))

             metric_key       metric_label metric_group  metric_order value_type direction
                  value          KTC Value        Value            10        num        up
               ds_value       DS 3D Value+        Value            11        num        up
           overall_tier       Overall Tier         Tier            20        num      down
        positional_tier    Positional Tier         Tier            21        num      down
               proj_1yr          Proj 1-Yr   Projection            30        num        up
               proj_3yr          Proj 3-Yr   Projection            31        num        up
               proj_5yr          Proj 5-Yr   Projection            32        num        up
              proj_10yr         Proj 10-Yr   Projection            33        num        up
                    avg        FP Avg Rank    Consensus            40        num      down
                   best       FP Best Rank    Consensus            41        num      down